In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

KNN

In [ ]:
from sklearn.metrics import f1_score, classification_report
import torch
import numpy as np
import time
import pandas as pd

X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

class TorchKNNClassifier:
    def __init__(self, n_neighbors=5, val_batch_size=256, train_batch_size=50000, device='cuda'):
        self.k = n_neighbors
        self.val_batch_size = val_batch_size
        self.train_batch_size = train_batch_size
        self.device = device
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        # Chuyển dữ liệu lên GPU một lần để tính toán
        if isinstance(X, pd.DataFrame): X = X.values
        if isinstance(y, pd.Series): y = y.values
        self.X_train = torch.tensor(X, dtype=torch.float32, device=self.device)
        self.y_train = torch.tensor(y, dtype=torch.long, device=self.device)

    def predict(self, X):
        if isinstance(X, pd.DataFrame): X = X.values
        X_val = torch.tensor(X, dtype=torch.float32, device=self.device)
        preds = []
        
        # Chia batch dữ liệu CẦN dự đoán 
        for i in range(0, len(X_val), self.val_batch_size):
            X_batch = X_val[i:i+self.val_batch_size]
            
            # Khởi tạo buffer chứa min distances. Mặc định là vô cực (inf)
            current_top_dists = torch.full((len(X_batch), self.k), float('inf'), device=self.device)
            current_top_indices = torch.zeros((len(X_batch), self.k), dtype=torch.long, device=self.device)
            
            # Phân tách X_train thành các khối nhỏ để lặp qua (Tránh tạo ma trận khổng lồ gây OOM)
            for j in range(0, len(self.X_train), self.train_batch_size):
                X_tr_batch = self.X_train[j:j+self.train_batch_size]
                
                # Tính khoảng cách Euclidean
                dists = torch.cdist(X_batch, X_tr_batch)
                
                # Mức độ cắt k
                k_chunk = min(self.k, dists.shape[1])
                chunk_top_dists, chunk_top_idx = torch.topk(dists, k_chunk, dim=1, largest=False)
                
                # Sửa index thành chỉ số toàn cục trong self.X_train
                chunk_top_idx += j
                
                # Gộp với k nhỏ nhất của các khối trước, sau đó lại top-k một lần nữa
                merged_dists = torch.cat([current_top_dists, chunk_top_dists], dim=1)
                merged_idx = torch.cat([current_top_indices, chunk_top_idx], dim=1)
                
                current_top_dists, best_k_pos = torch.topk(merged_dists, self.k, dim=1, largest=False)
                current_top_indices = torch.gather(merged_idx, 1, best_k_pos)

                # Dọn rác VRAM
                del dists, merged_dists, merged_idx
            
            # Lấy nhãn của k láng giềng gần nhất chung cuộc
            topk_labels = self.y_train[current_top_indices]
            
            # Dự đoán theo nhãn chiếm đa số (mode)
            mode_labels, _ = torch.mode(topk_labels, dim=1)
            preds.append(mode_labels.cpu().numpy())
            
        return np.concatenate(preds)

# Thử các giá trị k khác nhau
k_values = [7]

best_k = None
best_f1 = -1
best_model = None

print("Đang huấn luyện phân lớp KNN (Multi-batch Memory Efficient qua PyTorch) và tìm 'k' tốt nhất...")
start_time = time.time()

for k in k_values:
    # Sử dụng TorchKNNClassifier kết hợp cắt nhỏ ma trận, GPU sẽ an toàn
    knn = TorchKNNClassifier(n_neighbors=k, val_batch_size=256, train_batch_size=50000, device='cuda')
    knn.fit(X_train, y_train)
    
    y_valid_pred = knn.predict(X_valid)
    
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"k = {k:2d} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_k = k
        best_model = knn

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> K tốt nhất tìm được là k = {best_k} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test
print("\n--- Đánh giá Model Tốt nhất (k={}) trên tập TEST ---".format(best_k))
y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

SVM

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...")
start_time = time.time()

# Thử các giá trị alpha (L2 regularization)
alpha_values = [0.0001]

best_alpha = None
best_f1 = -1
best_svm_model = None

# Khối 1: Chuyển dữ liệu lên Numpy 
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đẩy sang Tensor CUDA
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
# Đối với Multi-class SVM, giữ nguyên nhãn số nguyên (0, 1, 2... 10)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Kiến trúc Linear SVM (Multi-class) bằng Pytorch
class TorchLinearSVM(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes) # Sửa output thành 11 classes
        
    def forward(self, x):
        return self.linear(x)

# PyTorch hỗ trợ sẵn MultiMarginLoss dùng để tối ưu Multi-class SVM
criterion = nn.MultiMarginLoss()

for alpha in alpha_values:
    model = TorchLinearSVM(input_dim, num_classes).to(device)
    
    # Khai báo Optimizer - sử dụng weight_decay thay thế L2 penalty
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=alpha)
    
    model.train()
    epochs = 100
    batch_size = 8192
    num_samples = X_train_t.shape[0]
    
    for epoch in range(epochs):
        indices = torch.randperm(num_samples, device=device)
        for i in range(0, num_samples, batch_size):
            idx = indices[i:i+batch_size]
            batch_X = X_train_t[idx]
            batch_y = y_train_t[idx]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            # Tính toán trên GPU với Multi-Class Hinge Loss
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
    
    # Validation step
    model.eval()
    with torch.no_grad():
        valid_outputs = model(X_valid_t)
        # Sử dụng argmax để lấy chỉ số có logit cao nhất (tương ứng với nhãn)
        y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
        
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_alpha = alpha
        best_svm_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên test
print("\n--- Đánh giá Model Multi-class SVM Tốt nhất (alpha={}) trên tập TEST ---".format(best_alpha))
best_svm_model.eval()
with torch.no_grad():
    test_outputs = best_svm_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...
alpha = 0.000100 | Validation Macro F1: 0.9659

=> Quá trình huấn luyện và tuning hoàn tất trong 269.34 giây.
=> Cấu hình tốt nhất: alpha = 0.0001 với Validation Macro F1: 0.9659

--- Đánh giá Model Multi-class SVM Tốt nhất (alpha=0.0001) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9930    0.9504    0.9712    246000
           1     0.9791    0.9970    0.9880    613218
           2     0.8844    0.9007    0.8925     92157
           3     0.9999    0.9958    0.9979    552847
           4     0.9954    0.9985    0.9969    275494
           5     0.9984    0.9970    0.9977    234000
           6     0.9997    0.9995    0.9996    689483
           7     0.9543    0.5720    0.7153     69052
           8     0.9953    0.9973    0.9963    157261
           9     0.9871    0.1957    0.3267      1962
          10     0.9908    0.950

Random Forest

In [3]:
from xgboost import XGBRFClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử trên các max_depth khác nhau
max_depth_values = [15]

best_depth = None
best_f1 = -1
best_rf_model = None

for depth in max_depth_values:
    # Sử dụng XGBRFClassifier với cấu hình device='cuda' để chạy trên GPU
    rf_model = XGBRFClassifier(
        n_estimators=100, 
        max_depth=depth,
        n_jobs=-1, 
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    rf_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính điểm Macro F1
    y_valid_pred = rf_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_rf_model = rf_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Random Forest Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_rf_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [19:05:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


max_depth = 15 | Validation Macro F1: 0.9337

=> Quá trình huấn luyện và tuning hoàn tất trong 169.39 giây.
=> Cấu hình tốt nhất: max_depth = 15 với Validation Macro F1: 0.9337

--- Đánh giá mô hình Random Forest Tốt nhất (max_depth=15) trên tập TEST ---
              precision    recall  f1-score   support

           0     1.0000    0.9990    0.9995    246000
           1     1.0000    0.9996    0.9998    613218
           2     0.8944    0.8267    0.8592     92157
           3     1.0000    0.9964    0.9982    552847
           4     1.0000    0.9865    0.9932    275494
           5     1.0000    0.9998    0.9999    234000
           6     0.9995    1.0000    0.9998    689483
           7     0.8966    0.5267    0.6636     69052
           8     0.9978    0.9998    0.9988    157261
           9     0.2149    0.9760    0.3522      1962
          10     0.1378    0.8574    0.2374      1360
          11     0.4656    0.9491    0.6248     25973
          12     1.0000    1.0000    1.000

Decision Tree

In [4]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import gc

print("Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

for col in X_train.columns:
    if X_train[col].dtype == 'float64':
        X_train[col] = X_train[col].astype(np.float32)
for col in X_valid.columns:
    if X_valid[col].dtype == 'float64':
        X_valid[col] = X_valid[col].astype(np.float32)
for col in X_test.columns:
    if X_test[col].dtype == 'float64':
        X_test[col] = X_test[col].astype(np.float32)

# Dọn dẹp rác bộ nhớ
gc.collect()

# Thử nghiệm các giá trị max_depth khác nhau
max_depth_values = [25, 30, 35, 40]

best_depth = None
best_f1 = -1
best_dt_model = None

for depth in max_depth_values:
    # Dùng XGBClassifier với n_estimators=1 để thiết lập giống một Decision Tree chạy bằng GPU
    dt_model = XGBClassifier(
        n_estimators=1,
        learning_rate=1.0,
        max_depth=depth,
        n_jobs=-1,
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    dt_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính Macro F1
    y_valid_pred = dt_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_dt_model = dt_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_dt_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...
max_depth = 25 | Validation Macro F1: 0.8805
max_depth = 30 | Validation Macro F1: 0.8805
max_depth = 35 | Validation Macro F1: 0.8805
max_depth = 40 | Validation Macro F1: 0.8805

=> Quá trình huấn luyện và tuning hoàn tất trong 107.77 giây.
=> Cấu hình tốt nhất: max_depth = 25 với Validation Macro F1: 0.8805

--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth=25) trên tập TEST ---
              precision    recall  f1-score   support

           0     1.0000    0.9990    0.9995    246000
           1     0.9996    0.9999    0.9997    613218
           2     0.8790    0.8250    0.8511     92157
           3     1.0000    0.9961    0.9981    552847
           4     1.0000    0.9944    0.9972    275494
           5     1.0000    0.9998    0.9999    234000
           6     0.9937    1.0000    0.9968    689483
           7     0.8953    0.5138    0.6529     69052
           8     0.9991    

Logistic Regression

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Logistic Regression (PyTorch - Data ở RAM, xử lý trên GPU qua từng batch) và tìm 'alpha' & 'penalty' tốt nhất...")
start_time = time.time()

# Thử nghiệm các alpha và penalty l1, l2
alpha_values = [1e-6]
penalty_values = ['l1']

best_alpha = None
best_penalty = None
best_f1 = -1
best_logreg_model = None

# Khối 1: Chuyển đổi toàn bộ mảng dữ liệu về Numpy (nếu đang ở pandas DataFrame)
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Sử dụng torch.from_numpy để KHÔNG copy thêm một bản sao trong RAM (tiết kiệm 9GB RAM)
# Giữ dữ liệu ở CPU, chỉ đưa batch lên GPU khi tính toán để tránh quá tải VRAM
device = 'cuda' if torch.cuda.is_available() else 'cpu'

X_train_t = torch.from_numpy(X_trn).float()
y_train_t = torch.from_numpy(y_trn).long()
X_valid_t = torch.from_numpy(X_val).float()
X_test_t = torch.from_numpy(X_tst).float()

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Định nghĩa mạng: 1 tầng Linear duy nhất + Hàm phạt Cross Entropy = Logistic Regression
class TorchLogisticRegression(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes)
        
    def forward(self, x):
        return self.linear(x)

for penalty in penalty_values:
    for alpha in alpha_values:
        model = TorchLogisticRegression(input_dim, num_classes).to(device)
        
        # Nếu l2 thì dùng tham số weight_decay tích hợp sẵn của Optimizer tự tính toán
        weight_decay = alpha if penalty == 'l2' else 0.0
        optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        epochs = 100
        batch_size = 8192
        num_samples = X_train_t.shape[0]
        
        for epoch in range(epochs):
            # Xáo trộn index trên CPU
            indices = torch.randperm(num_samples)
            
            for i in range(0, num_samples, batch_size):
                idx = indices[i:i+batch_size]
                # Chỉ đưa batch hiện tại lên GPU
                batch_X = X_train_t[idx].to(device)
                batch_y = y_train_t[idx].to(device)
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                # Nếu l1 thì cộng tay penalty L1 Norm
                if penalty == 'l1':
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + alpha * l1_norm
                    
                loss.backward()
                optimizer.step()
        
        # Validation step (Dự đoán theo batch để tránh OOM)
        model.eval()
        valid_preds = []
        with torch.no_grad():
            for i in range(0, X_valid_t.shape[0], batch_size):
                batch_X_val = X_valid_t[i:i+batch_size].to(device)
                valid_outputs = model(batch_X_val)
                batch_preds = torch.argmax(valid_outputs, dim=1).cpu().numpy()
                valid_preds.append(batch_preds)
        
        y_valid_pred = np.concatenate(valid_preds)
            
        current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
        print(f"penalty = {penalty:2s}, alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
        
        # Chọn model tốt nhất dựa trên Macro F1
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_alpha = alpha
            best_penalty = penalty
            best_logreg_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: penalty = {best_penalty}, alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print(f"\n--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty={best_penalty}, alpha={best_alpha}) trên tập TEST ---")
best_logreg_model.eval()
test_preds = []
with torch.no_grad():
    for i in range(0, X_test_t.shape[0], batch_size):
        batch_X_tst = X_test_t[i:i+batch_size].to(device)
        test_outputs = best_logreg_model(batch_X_tst)
        batch_preds = torch.argmax(test_outputs, dim=1).cpu().numpy()
        test_preds.append(batch_preds)

y_test_pred_final = np.concatenate(test_preds)
print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Logistic Regression (PyTorch - Data ở RAM, xử lý trên GPU qua từng batch) và tìm 'alpha' & 'penalty' tốt nhất...
penalty = l1, alpha = 0.000001 | Validation Macro F1: 0.9841

=> Quá trình huấn luyện và tuning hoàn tất trong 413.33 giây.
=> Cấu hình tốt nhất: penalty = l1, alpha = 1e-06 với Validation Macro F1: 0.9841

--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty=l1, alpha=1e-06) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9957    0.9988    0.9972    246000
           1     0.9994    0.9979    0.9986    613218
           2     0.9236    0.8835    0.9031     92157
           3     1.0000    0.9980    0.9990    552847
           4     0.9979    0.9986    0.9983    275494
           5     1.0000    0.9977    0.9989    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9169    0.6319    0.7482     69052
           8     0.9945    0.9997    0.9971    157261
           9     0.9

XGBoost

In [4]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp XGBoost (sử dụng GPU) với bộ tham số đã chọn...")
start_time = time.time()

xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,        # Tăng tốc độ học lên một chút để hội tụ tốt hơn
    max_depth=8,               # Giảm từ 10 xuống 8 để tránh chẻ quá sâu overfit vào một số lớp
    min_child_weight=2,        # Tăng lên 2 để yêu cầu node có đủ mẫu mới chia, giúp tăng Precision
    gamma=0.2,                 # Giảm chi phí phân nhánh
    reg_lambda=1.5,
    reg_alpha=0.5,
    subsample=0.8,         
    colsample_bytree=0.8,
    max_delta_step=2,
    random_state=42,
    eval_metric='mlogloss',
    tree_method="hist",
    device="cuda",
    early_stopping_rounds=100  # Chờ kiên nhẫn hơn
)

# Huấn luyện mô hình, sử dụng tập validation cho early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=100 
)

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện hoàn tất trong {train_time:.2f} giây.")

# Dự đoán và đánh giá trên tập validation
y_valid_pred = xgb_model.predict(X_valid)
valid_f1 = f1_score(y_valid, y_valid_pred, average='macro')
print(f"=> Validation Macro F1: {valid_f1:.4f}")

# Đánh giá trên tập test
print(f"\n--- Đánh giá mô hình XGBoost trên tập TEST ---")
y_test_pred = xgb_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp XGBoost (sử dụng GPU) với bộ tham số đã chọn...
[0]	validation_0-mlogloss:2.02356	validation_1-mlogloss:2.02367
[100]	validation_0-mlogloss:0.01180	validation_1-mlogloss:0.01297
[200]	validation_0-mlogloss:0.00099	validation_1-mlogloss:0.00203
[300]	validation_0-mlogloss:0.00074	validation_1-mlogloss:0.00191
[366]	validation_0-mlogloss:0.00067	validation_1-mlogloss:0.00195

=> Quá trình huấn luyện hoàn tất trong 173.46 giây.
=> Validation Macro F1: 0.9965

--- Đánh giá mô hình XGBoost trên tập TEST ---
              precision    recall  f1-score   support

           0     1.0000    0.9990    0.9995    246000
           1     0.9996    1.0000    0.9998    613218
           2     0.9329    0.8984    0.9153     92157
           3     0.9999    0.9990    0.9995    552847
           4     1.0000    1.0000    1.0000    275494
           5     1.0000    0.9999    0.9999    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9459    0.8042 